In [1]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import optuna
import lightgbm as lgb

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from churn.config import RAW_DIR, SUBMISSION_DIR
from churn.data_check import check_numeric_features
from churn.features import build_pipelines


In [2]:
train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

### 检查具体数值分布是否异常

In [3]:
check_numeric_features(train)

数值数据分布:


,MonthlyCharges,TotalCharges,tenure
count,594194.000000,594194.000000,594194.000000
mean,65.866223,2494.377057,36.577258
std,31.067444,2353.916710,25.061922
min,18.250000,18.800000,1.000000
25%,29.900000,639.650000,12.000000
50%,74.100000,1433.650000,35.000000
75%,90.800000,4263.800000,62.000000
max,118.750000,8684.800000,72.000000



绝对偏差分布 (TotalCharges - tenure × MonthlyCharges):


count    594194.000000
mean        -11.409798
std         298.747588
min       -7369.250000
25%         -70.550000
50%           0.000000
75%          60.250000
max        7308.300000
dtype: float64


相对偏差率分布:


count    594194.000000
mean          0.076265
std           0.326440
min           0.000000
25%           0.017362
50%           0.043992
75%           0.090056
max         103.923636
dtype: float64


各阈值下的人数分布


相对偏差率,人数,占比
0.100000,"129,266",21.75%
0.200000,"42,038",7.07%
0.500000,"5,994",1.01%
1.000000,624,0.11%
5.000000,151,0.03%


训练集有少量异常值，但有75%的数据偏差在9%以下且相对偏差大的样本很少,不影响整体

In [4]:
check_numeric_features(test)

数值数据分布:


,MonthlyCharges,TotalCharges,tenure
count,254655.000000,254655.000000,254655.000000
mean,66.130803,2544.617351,37.279716
std,31.104139,2348.124379,24.777625
min,18.300000,18.800000,1.000000
25%,30.150000,677.625000,13.000000
50%,74.350000,1468.750000,36.000000
75%,91.100000,4317.300000,62.000000
max,118.750000,8684.800000,72.000000



绝对偏差分布 (TotalCharges - tenure × MonthlyCharges):


count    254655.000000
mean        -10.737555
std         303.934525
min       -7103.750000
25%         -73.400000
50%           0.000000
75%          62.475000
max        7316.750000
dtype: float64


相对偏差率分布:


count    254655.000000
mean          0.079192
std           0.483737
min           0.000000
25%           0.018507
50%           0.045100
75%           0.091523
max         109.639773
dtype: float64


各阈值下的人数分布


相对偏差率,人数,占比
0.100000,"56,507",22.19%
0.200000,"18,458",7.25%
0.500000,"2,678",1.05%
1.000000,296,0.12%
5.000000,70,0.03%


测试集有少量异常值，但有75%的数据偏差在9%以下且相对偏差大的样本很少,不影响整体

### 数据预处理与特征工程

| 类型 | 列 | 取值数 |
|---|---|---|
| 二值 | Partner、Dependents、PaperlessBilling | 2 |
| 三值（无序） | InternetService、MultipleLines、Contract 以及 4 个服务列（含 No internet service） | 3 |
| 四值（无序） | PaymentMethod | 4 |

In [6]:
pipe_linear_item, pipe_tree_item = build_pipelines()

- 数值列:StandardScaler
- 类别列:OneHotEncoder
- 二值列:passthrough

### 模型构建与比较

- 训练集：80%
- 验证集：20%

In [7]:
X = train.drop(columns="Churn")
y = train["Churn"]

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
y_train = y_train.map({"Yes": 1, "No": 0})
y_val   = y_val.map({"Yes": 1, "No": 0})
# 参数初筛,看哪类模型更适合这份数据
linear_models = {
    "LogisticRegression（逻辑回归）":LogisticRegression(max_iter=1000)
}
tree_models = {
    "DecisionTreeClassifier（决策树）": DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "RandomForestClassifier（随机森林）": RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=42, n_jobs=5),
    "HistGB": HistGradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_jobs=-1, random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1)
}
pipelines = {"Dummy（基线）": DummyClassifier(strategy="prior")}  

In [45]:
for name, model in linear_models.items():
    pipelines[name] = Pipeline([
        ("prep", pipe_linear_item),
        ("model", model)
    ])
    
for name, model in tree_models.items():
    pipelines[name] = Pipeline([
        ("prep", pipe_tree_item),
        ("model", model)
    ])

In [46]:
cv = StratifiedKFold(
    n_splits=5, # 分层 5 折交叉验证
    shuffle=True, # 将原始数据的顺序随机打乱
    random_state=42
)

In [21]:
scoring = ["roc_auc", "average_precision", "f1"]
rows = []
for name, pipe in pipelines.items():
    res = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    rows.append({
        "模型": name,
        **{f"{s}_mean": res[f"test_{s}"].mean() for s in scoring},
        **{f"{s}_std":  res[f"test_{s}"].std()  for s in scoring},
    })
results = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
results

,模型,roc_auc_mean,average_precision_mean,f1_mean,roc_auc_std,average_precision_std,f1_std
6,XGBoost,0.915001,0.751411,0.675587,0.001090,0.002355,0.002381
7,LightGBM,0.914708,0.750075,0.673041,0.001033,0.002172,0.002565
5,HistGB,0.914614,0.749945,0.672617,0.001115,0.002318,0.001982
4,RandomForestClassifier（随机森林）,0.912690,0.744348,0.667746,0.001033,0.002336,0.002695
3,AdaBoost,0.909450,0.731548,0.669561,0.001105,0.003603,0.002047
2,DecisionTreeClassifier（决策树）,0.908267,0.724581,0.658972,0.001145,0.002395,0.002629
1,LogisticRegression（逻辑回归）,0.906700,0.722031,0.669155,0.001157,0.003074,0.002219
0,Dummy（基线）,0.500000,0.225209,0.000000,0.000000,0.000004,0.000000


### 模型优化与评估

通过调参交叉验证XGBoost和LightGBM	看谁的表现最好，适度调随机森林和逻辑回归融合备用

In [48]:
prep_pipeline = clone(pipe_tree_item)
X_train_matrix = prep_pipeline.fit_transform(X_train)
y_arr = y_train.values
folds = list(cv.split(X_train_matrix, y_arr))
X_val_matrix = prep_pipeline.transform(X_val)

In [49]:
def objective(trial):
    params = {
        "n_estimators": 2000,
        "learning_rate": 0.08, # 暂时固定为较大的学习率加快训练速度
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 50, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "tree_method": "hist",
        "eval_metric": "auc",           
        "early_stopping_rounds": 50,
        "n_jobs": -1,
        "random_state": 42,
    }

    scores, best_iters = [], []
    for step, (tr_idx, va_idx) in enumerate(folds):
        model = XGBClassifier(**params)
        model.fit(
            X_train_matrix[tr_idx], y_arr[tr_idx],
            eval_set=[(X_train_matrix[va_idx], y_arr[va_idx])],
            verbose=False,
        )
        p = model.predict_proba(X_train_matrix[va_idx])[:, 1]
        scores.append(roc_auc_score(y_arr[va_idx], p))
        best_iters.append(model.best_iteration + 1)

        trial.report(np.mean(scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("best_iter_mean", int(np.mean(best_iters)))
    return np.mean(scores)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=60, show_progress_bar=True)


[I 2026-11-19 16:06:00,523] A new study created in memory with name: no-name-991ff516-faf4-4be3-89c1-cf633a83c56d
Best trial: 0. Best value: 0.915894:   2%|▏         | 1/60 [01:39<1:37:39, 99.32s/it]

[I 2026-11-19 16:07:39,842] Trial 0 finished with value: 0.9158942251145517 and parameters: {'max_depth': 5, 'min_child_weight': 41.23206532618726, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'reg_lambda': 0.037766633271073366, 'reg_alpha': 0.004207053950287938, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.9158942251145517.


Best trial: 0. Best value: 0.915894:   3%|▎         | 2/60 [03:12<1:32:38, 95.84s/it]

[I 2026-11-19 16:09:13,248] Trial 1 finished with value: 0.9158285134526375 and parameters: {'max_depth': 9, 'min_child_weight': 10.502105436744278, 'subsample': 0.8540362888980227, 'colsample_bytree': 0.5102922471479012, 'reg_lambda': 38.69612257414268, 'reg_alpha': 2.1368329072358767, 'gamma': 1.0616955533913808}. Best is trial 0 with value: 0.9158942251145517.


Best trial: 0. Best value: 0.915894:   5%|▌         | 3/60 [06:17<2:09:29, 136.31s/it]

[I 2026-11-19 16:12:17,727] Trial 2 finished with value: 0.9157557149238598 and parameters: {'max_depth': 4, 'min_child_weight': 2.0492680115417348, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'reg_lambda': 0.39605150456850813, 'reg_alpha': 0.014618962793704957, 'gamma': 3.0592644736118975}. Best is trial 0 with value: 0.9158942251145517.


... (89 outputs left out)

[I 2026-11-19 18:07:26,268] Trial 47 finished with value: 0.9159734967851408 and parameters: {'max_depth': 4, 'min_child_weight': 1.6621725488818013, 'subsample': 0.8349953720350596, 'colsample_bytree': 0.9470729256870405, 'reg_lambda': 5.085316025924667, 'reg_alpha': 0.0010257642957638533, 'gamma': 0.31078311995072255}. Best is trial 38 with value: 0.9160491801739417.


Best trial: 38. Best value: 0.916049:  80%|████████  | 48/60 [2:03:21<30:50, 154.20s/it]


[W 2026-11-19 18:09:21,872] Trial 48 failed with parameters: {'max_depth': 3, 'min_child_weight': 3.465960926974487, 'subsample': 0.8213499320714297, 'colsample_bytree': 0.9765927020527705, 'reg_lambda': 7.723616868531182, 'reg_alpha': 0.0017051549559671914, 'gamma': 0.8563977451342342} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\optuna\study\_optimize.py", line 210, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\liang\AppData\Local\Temp\ipykernel_11248\660375737.py", line 22, in objective
    model.fit(
    ~~~~~~~~~^
        X_train_matrix[tr_idx], y_arr[tr_idx],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        eval_set=[(X_train_matrix[va_idx], y_arr[va_idx])],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        verbose=False,
        ^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\x

KeyboardInterrupt: 

In [50]:
print("best roc-auc :", study.best_value)
print("best params:", study.best_params)
print("avg best iters:", study.best_trial.user_attrs["best_iter_mean"])

best roc-auc : 0.9160491801739417
best params: {'max_depth': 3, 'min_child_weight': 1.8499842251284397, 'subsample': 0.9130491236562078, 'colsample_bytree': 0.9458048670145641, 'reg_lambda': 5.647395524814135, 'reg_alpha': 0.002040115167270367, 'gamma': 0.12964334666307942}
avg best iters: 1635


In [57]:
# 各参数的重要性，看哪些真的影响分数
display(optuna.importance.get_param_importances(study))

# 最近 10 个最好的 trial，看参数是否聚集
df = study.trials_dataframe().sort_values("value", ascending=False).head(10)
display(df[["value"] + [c for c in df.columns if c.startswith("params_")]])

{'gamma': 0.39465852330640727,
 'reg_alpha': 0.17719529717064939,
 'colsample_bytree': 0.14147283850524583,
 'reg_lambda': 0.11712203240014271,
 'min_child_weight': 0.08054832358376209,
 'max_depth': 0.07372197621645535,
 'subsample': 0.01528100881733744}

,value,params_colsample_bytree,params_gamma,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample
38,0.916049,0.945805,0.129643,3,1.849984,0.002040,5.647396,0.913049
39,0.916047,0.933356,0.098313,3,2.177618,0.002048,13.051167,0.860322
44,0.916030,0.991878,0.080268,3,2.180064,0.002088,9.930017,0.785142
21,0.916027,0.872905,0.769425,4,2.923288,0.001148,19.202776,0.769532
29,0.916026,0.944567,0.030885,3,3.943255,0.001038,10.788515,0.822432
36,0.916025,0.912278,0.758529,3,2.803045,0.006586,6.006742,0.882871
31,0.916016,0.996242,0.623674,3,2.324901,0.003949,16.073447,0.906768
26,0.916015,0.646064,1.174161,4,3.418745,0.001693,9.470895,0.825111
30,0.916012,0.949480,0.660888,3,1.867651,0.004080,21.505251,0.904198
43,0.916006,0.982030,0.331695,3,2.427031,0.007371,4.451686,0.899598


In [62]:
best = study.best_params.copy()
final_params = {
    **best,
    "n_estimators": 5000,
    "learning_rate": 0.05,
    "tree_method": "hist",
    "eval_metric": "auc",
    "early_stopping_rounds": 100,
    "n_jobs": -1,
    "random_state": 42,
}

scores_auc, scores_ap, iters = [], [], []
oof = np.zeros(len(y_arr))
for tr_idx, va_idx in folds:
    m = XGBClassifier(**final_params)
    m.fit(X_train_matrix[tr_idx], y_arr[tr_idx],
          eval_set=[(X_train_matrix[va_idx], y_arr[va_idx])],
          verbose=False)
    p = m.predict_proba(X_train_matrix[va_idx])[:, 1]
    oof[va_idx] = p
    scores_auc.append(roc_auc_score(y_arr[va_idx], p))
    iters.append(m.best_iteration + 1)

In [63]:
print("AUC", np.mean(scores_auc), "iters", iters)

AUC 0.9160886818926901 iters [3113, 2941, 2735, 2904, 2346]


In [66]:
def objective_lgb(trial):
    params = {
        "n_estimators": 2000,
        "learning_rate": 0.08,
        "num_leaves": trial.suggest_int("num_leaves", 8, 128, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 5),
        "max_depth": -1,
        "n_jobs": -1,
        "random_state": 42,
        "verbose": -1,
    }

    scores, best_iters = [], []
    for step, (tr_idx, va_idx) in enumerate(folds):
        model = LGBMClassifier(**params)
        model.fit(
            X_train_matrix[tr_idx], y_arr[tr_idx],
            eval_set=[(X_train_matrix[va_idx], y_arr[va_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        p = model.predict_proba(X_train_matrix[va_idx])[:, 1]
        scores.append(roc_auc_score(y_arr[va_idx], p))
        best_iters.append(model.best_iteration_)

        trial.report(np.mean(scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("best_iter_mean", int(np.mean(best_iters)))
    return np.mean(scores)

study_lgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

[I 2026-11-19 18:52:19,380] A new study created in memory with name: no-name-a339d757-05b8-4da2-95c0-82c32bf0ed9e
  0%|          | 0/30 [00:00<?, ?it/s]C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\

[I 2026-11-19 18:52:45,441] Trial 0 finished with value: 0.9159003214281318 and parameters: {'num_leaves': 22, 'min_child_samples': 427, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'reg_lambda': 0.037766633271073366, 'reg_alpha': 0.004207053950287938, 'min_split_gain': 0.2904180608409973}. Best is trial 0 with value: 0.9159003214281318.


C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

[I 2026-11-19 18:53:32,012] Trial 1 finished with value: 0.915767299869246 and parameters: {'num_leaves': 88, 'min_child_samples': 137, 'subsample': 0.8540362888980227, 'colsample_bytree': 0.5102922471479012, 'reg_lambda': 38.69612257414268, 'reg_alpha': 2.1368329072358767, 'min_split_gain': 1.0616955533913808}. Best is trial 0 with value: 0.9159003214281318.


C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

[I 2026-11-19 18:54:17,919] Trial 2 finished with value: 0.9152623443408116 and parameters: {'num_leaves': 13, 'min_child_samples': 35, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'reg_lambda': 0.39605150456850813, 'reg_alpha': 0.014618962793704957, 'min_split_gain': 3.0592644736118975}. Best is trial 0 with value: 0.9159003214281318.


... (50 outputs left out)

C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

[I 2026-11-19 19:12:22,477] Trial 28 finished with value: 0.9160338816950262 and parameters: {'num_leaves': 11, 'min_child_samples': 181, 'subsample': 0.8837261886524295, 'colsample_bytree': 0.5060344394116074, 'reg_lambda': 32.69402111193022, 'reg_alpha': 0.006670704282446171, 'min_split_gain': 0.1975662472123375}. Best is trial 18 with value: 0.9160724536596326.


C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

[I 2026-11-19 19:13:14,912] Trial 29 finished with value: 0.9159919610235081 and parameters: {'num_leaves': 12, 'min_child_samples': 185, 'subsample': 0.9349817176233468, 'colsample_bytree': 0.506826637503104, 'reg_lambda': 3.4072624518350545, 'reg_alpha': 0.001211537534105553, 'min_split_gain': 0.08783111549391662}. Best is trial 18 with value: 0.9160724536596326.


In [67]:
print("best roc-auc :", study_lgb.best_value)
print("best params:", study_lgb.best_params)
print("avg best iters:", study_lgb.best_trial.user_attrs["best_iter_mean"])

best roc-auc : 0.9160724536596326
best params: {'num_leaves': 10, 'min_child_samples': 85, 'subsample': 0.9281577616934087, 'colsample_bytree': 0.5537981064820668, 'reg_lambda': 35.50228641308715, 'reg_alpha': 0.001317653231802295, 'min_split_gain': 0.18102084748890865}
avg best iters: 1537


In [69]:
best_params_lgb = study_lgb.best_params.copy()
final_params_lgb = {
    **best_params_lgb,
    "learning_rate": 0.05,
    "n_estimators": 5000,
    "subsample_freq": 1,
    "max_depth": -1,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
}

scores_auc_lgb, iters_lgb = [],[]
oof_lgb = np.zeros(len(y_arr))
for tr_idx, val_idx in folds:
    m = LGBMClassifier(**final_params_lgb)
    m.fit(
        X_train_matrix[tr_idx], y_arr[tr_idx],
        eval_set=[(X_train_matrix[val_idx], y_arr[val_idx])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    p = m.predict_proba(X_train_matrix[val_idx])[:, 1]
    oof_lgb[val_idx] = p
    scores_auc_lgb.append(roc_auc_score(y_arr[val_idx], p))
    iters_lgb.append(m.best_iteration_)

C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\liang\Desktop\customer-churn\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  ev

In [71]:
print("AUC", np.mean(scores_auc_lgb), "iters", iters_lgb)
print("OOF AUC", roc_auc_score(y_arr, oof_lgb))

AUC 0.9160907327620155 iters [2256, 3092, 2436, 3112, 3013]
OOF AUC 0.9160861955621774


In [73]:
print("相关系数:", np.corrcoef(oof, oof_lgb)[0, 1])
print("XGB AUC  :", roc_auc_score(y_arr, oof))
print("LGB AUC  :", roc_auc_score(y_arr, oof_lgb))
print("平均融合 AUC:", roc_auc_score(y_arr, (oof + oof_lgb) / 2))

相关系数: 0.997588193416787
XGB AUC  : 0.9160840463586853
LGB AUC  : 0.9160861955621774
平均融合 AUC: 0.9162592607066141


In [74]:
prep_lin = clone(pipe_linear_item)
X_lin = prep_lin.fit_transform(X_train)

oof_lr = np.zeros(len(y_arr))
for tr_idx, va_idx in folds:
    lr = LogisticRegression(max_iter=2000, C=1.0)
    lr.fit(X_lin[tr_idx], y_arr[tr_idx])
    oof_lr[va_idx] = lr.predict_proba(X_lin[va_idx])[:, 1]

print("LR AUC:", roc_auc_score(y_arr, oof_lr))
print("与XGB相关:", np.corrcoef(oof, oof_lr)[0, 1])

LR AUC: 0.9066957647509796
与XGB相关: 0.9653979655395991


In [76]:
base = (oof + oof_lgb) / 2
print("XGB+LGB 基准:", roc_auc_score(y_arr, base))
for w in [0, 0.1, 0.2, 0.3, 0.4]:
    blend = (1 - w) * base + w * oof_lr
    print(f"w={w}: AUC={roc_auc_score(y_arr, blend):.6f}  ")

XGB+LGB 基准: 0.9162592607066141
w=0: AUC=0.916259  
w=0.1: AUC=0.916178  
w=0.2: AUC=0.915925  
w=0.3: AUC=0.915499  
w=0.4: AUC=0.914894  


In [78]:
from scipy.stats import rankdata
r = lambda x: rankdata(x) / len(x)
base_r = (r(oof) + r(oof_lgb)) / 2
for w in [0, 0.1, 0.2, 0.3, 0.4]:
    print(w, roc_auc_score(y_arr, (1 - w) * base_r + w * r(oof_lr)))

0 0.9162594180309779
0.1 0.916161902516337
0.2 0.9158668219671742
0.3 0.9153748939809234
0.4 0.9146891427462819


经验证发现，用逻辑回归融合后roc-auc不增反降，所以不考虑将其融合

In [81]:
X_val_matrix = prep_pipeline.transform(X_val)  

# 用全量训练集训练两个模型
n_xgb = int(np.mean(iters) * 1.1)
n_lgb = int(np.mean(iters_lgb) * 1.1)

xgb_final_params = {k: v for k, v in final_params.items() if k != "early_stopping_rounds"}
xgb_final_params["n_estimators"] = n_xgb
xgb_final = XGBClassifier(**xgb_final_params).fit(X_train_matrix, y_arr)

lgb_final_params = {**final_params_lgb, "n_estimators": n_lgb}
lgb_final = LGBMClassifier(**lgb_final_params).fit(X_train_matrix, y_arr)

# 3 在 X_val 上评估
p_x = xgb_final.predict_proba(X_val_matrix)[:, 1]
p_l = lgb_final.predict_proba(X_val_matrix)[:, 1]
for name, p in [("XGB", p_x), ("LGB", p_l), ("XGB+LGB", (p_x + p_l) / 2)]:
    print(name, "AUC", roc_auc_score(y_val, p))

XGB AUC 0.9166547536146614
LGB AUC 0.9166730938265004
XGB+LGB AUC 0.9168159000993376


In [ ]:
X_full = train.drop(columns="Churn")
y_full = train["Churn"].map({"Yes": 1, "No": 0}).values

prep_final = clone(pipe_tree_item)
X_full_mat = prep_final.fit_transform(X_full)
X_test_mat = prep_final.transform(test)

xgb_sub = XGBClassifier(**xgb_final_params).fit(X_full_mat, y_full)
lgb_sub = LGBMClassifier(**lgb_final_params).fit(X_full_mat, y_full)

pred = (xgb_sub.predict_proba(X_test_mat)[:, 1] +
        lgb_sub.predict_proba(X_test_mat)[:, 1]) / 2

sample = pd.read_csv(RAW_DIR / "sample_submission.csv")
sub = pd.DataFrame({"id": test["id"], "Churn": pred})
SUBMISSION_DIR.mkdir(exist_ok=True)
sub.to_csv(SUBMISSION_DIR / "submission.csv", index=False)